## TEST

Portability: storage is resolved by `tools/project_paths.py`. Accepted outputs are read-only. Use `04D_experiments.ipynb` for the Colab bootstrap and non-training smoke test. Saved cell outputs below are historical.


In [6]:
# Each notebook kernel discovers its own runtime checkout.
from pathlib import Path, PureWindowsPath
import os, subprocess, sys
_repo_hint = os.environ.get("EDGEAI_REPO_ROOT")
if _repo_hint and os.name != "nt" and PureWindowsPath(_repo_hint).drive:
    raise RuntimeError("EDGEAI_REPO_ROOT must name the checkout in this runtime, not a Windows drive")
_start = Path(_repo_hint).expanduser().resolve() if _repo_hint else Path.cwd().resolve()
_candidates = [_start, *_start.parents, Path("/content/edge-ai")]
_repo = next((p for p in _candidates if (p / "tools/project_paths.py").is_file()), None)
if _repo is None:
    try:
        import google.colab
    except ImportError:
        raise RuntimeError("Set EDGEAI_REPO_ROOT to your local checkout and rerun this cell") from None
    _repo = Path(_repo_hint or "/content/edge-ai").expanduser().resolve()
    if _repo.exists():
        raise RuntimeError(f"{_repo} exists but is not an EdgeAI checkout; set EDGEAI_REPO_ROOT to the correct checkout")
    subprocess.run(["git", "clone", "https://github.com/ancaranovik/Mosquito_Wingbeat_v1.git", str(_repo)], check=True)
if not (_repo / "tools/notebook_runtime.py").is_file():
    raise RuntimeError("Checkout is missing notebook_runtime.py. Pull the latest GitHub commit in this runtime, then rerun this cell")
sys.path.insert(0, str(_repo / "tools"))
from notebook_runtime import initialize
PATHS = initialize(_repo)
from project_paths import REPO_ROOT, ProjectPaths, resolve_path, logical_relative
import sys, torch, os

print("Python:", sys.version)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
print("cwd:", os.getcwd())

Repository: /content/edge-ai
Data: /content/drive/MyDrive/EdgeAI
Immutable baseline: /content/drive/MyDrive/EdgeAI/fixed/baseline/stage04
Future results: /content/drive/MyDrive/EdgeAI/experiments/<experiment_id>
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
CUDA: True
GPU: Tesla T4
cwd: /content/edge-ai


# 04A — Frozen Log-Mel training

Four from-scratch classifiers in a controlled comparison of two frozen acoustic frontend configurations.
No frozen data/frontend artifact is rewritten. Read [the full protocol](../../docs/STAGE04.md) before training.
Install `requirements-stage04.txt` in a dedicated CPython 3.12 kernel. Training is deliberately opt-in below;
all preflight checks run first. This notebook uses shared code with its companion branch.


In [7]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
PROJECT = REPO_ROOT
sys.path.insert(0, str(PROJECT / "tools"))
import importlib
import stage04_data, model_zoo, training_common
for module in (stage04_data, model_zoo, training_common):
    importlib.reload(module)
import pandas as pd
from IPython.display import display
from model_zoo import FAMILIES, architecture_config, build_model
from training_common import CONFIG, run_branch, compare_saved_results, RUNS
from stage04_data import BRANCHES, CLASSES, ROOT, prepare_caches, read_json

FRONTEND = "03A"
RUN_TRAINING = False


## Fixed protocol and architecture definitions

CPU PyTorch; seed 42; TRAIN-only inverse-frequency weights; weighted cross-entropy; Adam 0.001;
batch 64; constant learning rate; at most 50 epochs; patience 8; lowest weighted validation loss
selects the checkpoint (earliest exact tie). No input normalization or augmentation.
TEST is evaluated only after the selected checkpoint and its SHA256 are saved.


In [8]:
display(pd.Series(CONFIG, name="Fixed Stage-04 protocol"))
architecture_rows = []
for family in FAMILIES:
    width = BRANCHES[FRONTEND]["shape"][1]
    model = build_model(family, width)
    architecture_rows.append({"family": family, "input": [96, width],
                              "parameters": sum(p.numel() for p in model.parameters())})
    print(family, architecture_config(family, width))
display(pd.DataFrame(architecture_rows))


seed                                                                            42
device                                                                         cpu
dtype                                                                      float32
cpu_threads                                                                      4
batch_size                                                                      64
max_epochs                                                                      50
early_stopping_patience                                                          8
selection                                minimum_validation_weighted_cross_entropy
min_delta                                                                      0.0
tie_rule                                                            earliest_epoch
optimizer                                                                     Adam
learning_rate                                                                0.001
beta

small_cnn {'channels': [16, 32, 64], 'kernel': [3, 3], 'stride': [1, 1], 'max_pool_after_first_two': [2, 2], 'head': 'global_mean_time_frequency -> dropout -> linear4', 'family': 'small_cnn', 'input_shape': [1, 96, 64], 'classes': 4, 'activation': 'ReLU', 'dropout': 0.2, 'convolution_bias': False, 'convolution_padding': 'symmetric k//2', 'batch_norm': {'eps': 1e-05, 'momentum': 0.1, 'affine': True}, 'linear_bias': True, 'initialization': 'PyTorch default reset_parameters; seed 42', 'pretrained': False}
ds_cnn {'stem_channels': 32, 'stem_kernel': [3, 3], 'stem_stride': [2, 2], 'blocks': 4, 'channels': 32, 'depthwise_kernel': [3, 3], 'block_stride': [1, 1], 'pointwise_kernel': [1, 1], 'head': 'global_mean_time_frequency -> dropout -> linear4', 'family': 'ds_cnn', 'input_shape': [1, 96, 64], 'classes': 4, 'activation': 'ReLU', 'dropout': 0.2, 'convolution_bias': False, 'convolution_padding': 'symmetric k//2', 'batch_norm': {'eps': 1e-05, 'momentum': 0.1, 'affine': True}, 'linear_bias': Tr

,family,input,parameters
0,small_cnn,"[96, 64]",23668
1,ds_cnn,"[96, 64]",6244
2,tc_resnet8,"[96, 64]",65940
3,cnn_lstm,"[96, 64]",13428


## Mandatory preflight and paired caches

Authenticates all accepted data/source/WAV identities, frozen audits, labels, complete grid and split membership.
Builds separate Stage-04 caches if missing; checks every cached tensor and exact example alignment across both
branches before training. Do not repair failures by updating frozen pins, dropping examples or changing DSP.
Historical 00–02 notebook execution-hash discrepancies, if present, are reported explicitly; see protocol notes.


In [9]:
caches, evidence = prepare_caches()
display(pd.DataFrame({"species": CLASSES, "TRAIN examples": evidence["train_counts"],
                     "class weight": [evidence["class_weights"][s] for s in CLASSES]}))
display(pd.Series(evidence["counts"], name="Frozen split counts"))
for note in evidence["historical_review_notes"]:
    print(note)
assert evidence["identical_branch_example_uids_labels_splits"]
print("PASS: paired caches, 32,645 identities, four labels, fixed splits and no source-name leakage.")
del caches  # run_branch independently authenticates and opens its own read-only maps


,species,TRAIN examples,class weight
0,an arabiensis,10457,0.540619
1,an funestus ss,5128,1.102428
2,an squamosus,1426,3.964411
3,culex pipiens complex,5602,1.009149


train         22613
validation     4937
test           5095
Name: Frozen split counts, dtype: int64

{'notebook': 'notebooks/current/00_prepare_dataset.ipynb', 'historical_sha256': '2ee89d7b424c517b165a0c9a56f785d6540df79bd487abd1bd8be9c922cc95d3', 'current_frozen_sha256': '0d6115c2c04daa6922011fc25a6a2fcf0c40404f960db69da84244e0d1ba2478', 'note': 'Historical whole-notebook hash differs; handled as provenance only. Scientific AST/artifact gates are checked separately.'}
{'notebook': 'notebooks/current/01_dataset_audit.ipynb', 'historical_sha256': 'd8fdb548b5c3c6b4ce6a0d13723da692358adee77f49b709dddd79a7ab6babf2', 'current_frozen_sha256': 'fa110b1d4abf15fec9b5e34b091bda71867e82b0654a62060eff28a0ac47229e', 'note': 'Historical whole-notebook hash differs; handled as provenance only. Scientific AST/artifact gates are checked separately.'}
{'notebook': 'notebooks/current/02_segmentation.ipynb', 'historical_sha256': '05b65e69d225133441cce7ebf123124f7c4080e671a252c3fcb63e08febeb495', 'current_frozen_sha256': 'f22aede332ac8d6898cfcdecd82abf9bce17113b5ff18605c4430fd4c70242fc', 'note': 'Histori

## Execute four full experiments

Set `RUN_TRAINING = True` in the setup cell, then rerun the training cell when ready.
Do not alter the protocol for one frontend. Completed runs are authenticated and reused;
interrupted runs remain visible and require investigation. Training does not receive TEST loaders.


In [10]:
if RUN_TRAINING:
    branch_results = run_branch(FRONTEND)
    display(branch_results)
else:
    print("READY; full training was not launched (RUN_TRAINING=False).")


READY; full training was not launched (RUN_TRAINING=False).


## Interpretation

Primary results are example-level classification. Windows from a recording are correlated and are not
independent biological mosquitoes. Secondary source metrics average window probabilities by source name.
Use 04C only after both branches finish. Model size is serialized PyTorch size, not hardware Flash/RAM.
Stage 05 will assess quantization, latency and hardware compatibility; this notebook does none of those.
